<p style="font-family: 'Courier New', Courier, monospace; font-size: 30px; font-weight: bold; color: blue;  text-align: left;">
SVR for Regression 
</p>

In [ ]:
# Imports - core utils, data wrangling, ML, optimization, and plotting

# Standard library
import os
import pickle
import re
import time
import warnings

# Data handling
import numpy as np
import pandas as pd
from IPython.display import display

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress

# SVR pipeline and preprocessing
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.svm import SVR

# Model selection and evaluation
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error,
    median_absolute_error,
)
from sklearn.model_selection import PredefinedSplit, learning_curve

# Bayesian optimization
from skopt import BayesSearchCV
from skopt.space import Integer, Real, Categorical

# Suppress known compatibility/noise warnings
warnings.filterwarnings("ignore", category=UserWarning, module="skopt")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, message="X does not have valid feature names")


<p style="font-family: 'Courier New', Courier, monospace; font-size: 30px; font-weight: bold; color: blue;  text-align: left;">
Load the same standardized dataset and folds
</p>

In [ ]:
# Paths - aligned with the MLR, RF, XGBoost, LightGBM, and KNN notebooks
TRAIN_CSV = "Data_Files/train.csv"
TEST_CSV  = "Data_Files/test.csv"
FOLDS_NPY = "Data_Files/train_folds.npy"

MODELS_DIR     = "Models"
FIGURES_DIR    = "Figures"
CV_RESULTS_DIR = "CV_Results"
RESIDUALS_DIR  = "Residuals"

for directory in [MODELS_DIR, FIGURES_DIR, CV_RESULTS_DIR, RESIDUALS_DIR]:
    os.makedirs(directory, exist_ok=True)

CV_RESULTS_PATH = os.path.join(CV_RESULTS_DIR, "svr_bayes_cv_results.csv")
C_SWEEP_PATH    = os.path.join(CV_RESULTS_DIR, "svr_c_sweep_cv.csv")
MODEL_PATH      = os.path.join(MODELS_DIR, "svr_final_model.pkl")

# Load time-aware train/test split and predefined train folds
df_train = pd.read_csv(TRAIN_CSV)
df_test  = pd.read_csv(TEST_CSV)
fold_assignments = np.load(FOLDS_NPY)

# Optional datetime parsing for reporting/saving
for df in [df_train, df_test]:
    if "time" in df.columns:
        df["time"] = pd.to_datetime(df["time"], errors="coerce")

# Feature/target setup
feature_names = [
    "distance", "frequency", "c_walls", "w_walls",
    "co2", "humidity", "pm25", "pressure", "temperature", "snr"
]
target_col = "PL"

required_cols = [target_col, "device_id", *feature_names]
missing_train = [c for c in required_cols if c not in df_train.columns]
missing_test  = [c for c in required_cols if c not in df_test.columns]

if missing_train or missing_test:
    raise ValueError(
        f"Missing columns | train: {missing_train} | test: {missing_test}"
    )

if len(fold_assignments) != len(df_train):
    raise ValueError(
        f"fold_assignments length ({len(fold_assignments)}) does not match "
        f"df_train length ({len(df_train)})"
    )

# Train/test matrices
X_train_df = df_train[feature_names].copy()
X_test_df  = df_test[feature_names].copy()

X_train = X_train_df.to_numpy()
y_train = df_train[target_col].astype(float).to_numpy()

X_test = X_test_df.to_numpy()
y_test = df_test[target_col].astype(float).to_numpy()

ps = PredefinedSplit(fold_assignments)

print(f"Training samples: {len(df_train)}, Test samples: {len(df_test)}")
if "time" in df_train.columns and "time" in df_test.columns:
    print(f"Train window: {df_train.time.min()} -> {df_train.time.max()}")
    print(f"Test window:  {df_test.time.min()} -> {df_test.time.max()}")

unique, counts = np.unique(fold_assignments, return_counts=True)
print("Fold sizes:", dict(zip(unique.astype(int), counts.astype(int))))


In [ ]:
# Optional subset for fast SVR experiments. Set USE_SUBSET=False for full research runs.
USE_SUBSET = True
DATA_FRACTION = 0.30  # fraction of rows to keep from each fold, preserving time order

if USE_SUBSET:
    fold_ids = np.unique(fold_assignments[fold_assignments != -1])
    keep_mask = np.zeros(len(df_train), dtype=bool)
    kept_counts = {}

    for fold_id in fold_ids:
        idx = np.flatnonzero(fold_assignments == fold_id)
        if idx.size == 0:
            kept_counts[int(fold_id)] = 0
            continue

        keep_n = max(1, int(np.ceil(idx.size * DATA_FRACTION)))
        step = max(1, int(np.floor(idx.size / keep_n)))
        selected = idx[::step][:keep_n]
        keep_mask[selected] = True
        kept_counts[int(fold_id)] = selected.size

    if np.any(fold_assignments == -1):
        idx = np.flatnonzero(fold_assignments == -1)
        if idx.size > 0:
            keep_n = max(1, int(np.ceil(idx.size * DATA_FRACTION)))
            step = max(1, int(np.floor(idx.size / keep_n)))
            selected = idx[::step][:keep_n]
            keep_mask[selected] = True
            kept_counts[-1] = selected.size

    df_train = df_train.iloc[keep_mask].copy()
    fold_assignments = fold_assignments[keep_mask]

    print("Using subset per fold for SVR:", kept_counts)
    print(f"Train rows kept: {len(df_train)}, Test rows kept: {len(df_test)}")
else:
    print("Using full dataset")

if len(df_train) == 0 or len(df_test) == 0:
    raise ValueError("Subset produced empty data; increase DATA_FRACTION or disable USE_SUBSET.")

if len(fold_assignments) != len(df_train):
    raise ValueError(
        f"fold_assignments length ({len(fold_assignments)}) does not match "
        f"df_train length ({len(df_train)})"
    )

# Refresh matrices/split object after optional subsetting
X_train_df = df_train[feature_names].copy()
X_test_df  = df_test[feature_names].copy()

X_train = X_train_df.to_numpy()
y_train = df_train[target_col].astype(float).to_numpy()

X_test = X_test_df.to_numpy()
y_test = df_test[target_col].astype(float).to_numpy()

ps = PredefinedSplit(fold_assignments)

cv_fold_ids = fold_assignments[fold_assignments != -1]
if cv_fold_ids.size:
    unique, counts = np.unique(cv_fold_ids, return_counts=True)
    print("CV fold sizes:", dict(zip(unique.astype(int), counts.astype(int))))
else:
    print("No validation fold labels found in fold_assignments.")


<p style="font-family: 'Courier New', Courier, monospace; font-size: 30px; font-weight: bold; color: blue;  text-align: left;">
Define SVR pipeline + search space
</p>

Why a pipeline? SVR is highly scale‑sensitive (C and ε depend on the target scale and feature magnitudes). We standardize features inside CV. I also let the search toggle the scaler (Standard vs Robust) to hedge against outliers.

In [ ]:
def create_svr_pipeline():
    return Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("svr", SVR(cache_size=1000)),
    ])

space_rbf = {
    "scaler": Categorical([StandardScaler(), RobustScaler(quantile_range=(10.0, 90.0))]),
    "svr__kernel": Categorical(["rbf"]),
    "svr__C": Real(1e-2, 1e3, prior="log-uniform"),
    "svr__epsilon": Real(1e-3, 2e-1, prior="log-uniform"),
    "svr__gamma": Real(1e-4, 1e0, prior="log-uniform"),
}

space_poly = {
    "scaler": Categorical([StandardScaler(), RobustScaler(quantile_range=(10.0, 90.0))]),
    "svr__kernel": Categorical(["poly"]),
    "svr__C": Real(1e-2, 1e3, prior="log-uniform"),
    "svr__epsilon": Real(1e-3, 2e-1, prior="log-uniform"),
    "svr__gamma": Real(1e-5, 1e-1, prior="log-uniform"),
    "svr__degree": Integer(2, 4),
    "svr__coef0": Real(0.0, 1.0),
}

search_spaces = [
    (space_rbf, 20),
    (space_poly, 10),
]


<p style="font-family: 'Courier New', Courier, monospace; font-size: 30px; font-weight: bold; color: blue;  text-align: left;">
Bayesian optimization with PredefinedSplit + two metrics
</p>

In [ ]:
# Setup for multi-metric scoring
scoring = {"neg_root_mean_squared_error": "neg_root_mean_squared_error", "r2": "r2"}

# PredefinedSplit for time-ordered CV
ps = PredefinedSplit(fold_assignments)

# Bayesian optimization, scoring on both RMSE and R2
bayes_cv_svr = BayesSearchCV(
    estimator=create_svr_pipeline(),
    search_spaces=search_spaces,
    n_iter=25,
    scoring=scoring,
    refit="neg_root_mean_squared_error",
    n_jobs=12,
    cv=ps,
    random_state=42,
    verbose=1,
    n_points=3,
    optimizer_kwargs={"n_initial_points": 12, "acq_func": "gp_hedge"},
)

print(
    f"Starting Bayesian optimization with {bayes_cv_svr.n_iter} iterations "
    f"and {ps.get_n_splits()}-fold cross-validation per candidate...\n"
)

t0 = time.time()
bayes_cv_svr.fit(X_train_df, y_train)
t1 = time.time()
print(f"\nBayesian optimization complete in {(t1 - t0) / 60:.2f} minutes.")

best_svr_params = bayes_cv_svr.best_params_
print("Best SVR Parameters Found:", best_svr_params)

bayes_results_svr = pd.DataFrame(bayes_cv_svr.cv_results_).copy()
print(f"\nTried {bayes_results_svr.shape[0]} SVR configurations.")

bayes_results_svr.to_csv(CV_RESULTS_PATH, index=False)
print(f"\nSaved BayesSearchCV cv_results_ to: {CV_RESULTS_PATH}")


<p style="font-family: 'Courier New', Courier, monospace; font-size: 30px; font-weight: bold; color: blue;  text-align: left;">
Summarize best CV results per kernel (for plotting & reporting)
</p>

In [ ]:
# Load saved CV-search results and derive the best SVR parameters robustly
legacy_cv_result_paths = [
    os.path.join("SVR", "Results", "bayes_cv_results.csv"),
    os.path.join("Results", "bayes_cv_results.csv"),
]

if os.path.exists(CV_RESULTS_PATH):
    bayes_results_svr = pd.read_csv(CV_RESULTS_PATH)
else:
    bayes_results_svr = None
    for legacy_path in legacy_cv_result_paths:
        if os.path.exists(legacy_path):
            bayes_results_svr = pd.read_csv(legacy_path)
            bayes_results_svr.to_csv(CV_RESULTS_PATH, index=False)
            print(f"Migrated legacy CV results to: {CV_RESULTS_PATH}")
            break

    if bayes_results_svr is None:
        raise FileNotFoundError(
            f"No CV results found. Run the BayesSearchCV cell or provide {CV_RESULTS_PATH}."
        )

if bayes_results_svr.empty:
    raise ValueError("Loaded CV results are empty.")

required_cv_cols = [
    "mean_test_neg_root_mean_squared_error",
    "std_test_neg_root_mean_squared_error",
    "mean_test_r2",
    "std_test_r2",
    "param_svr__kernel",
]
missing_cv_cols = [c for c in required_cv_cols if c not in bayes_results_svr.columns]
if missing_cv_cols:
    raise KeyError(f"Missing expected CV result columns: {missing_cv_cols}")

# Best overall configuration by validation RMSE
best_idx = bayes_results_svr["mean_test_neg_root_mean_squared_error"].idxmax()
best_row = bayes_results_svr.loc[best_idx]


def _parse_svr_param(name, value):
    if pd.isna(value) or value == "":
        return None

    text = str(value)

    if name == "scaler":
        if "RobustScaler" in text:
            return RobustScaler(quantile_range=(10.0, 90.0))
        return StandardScaler()
    if name in {"svr__C", "svr__epsilon", "svr__gamma", "svr__coef0"}:
        return float(value)
    if name == "svr__degree":
        return int(round(float(value)))
    if name == "svr__kernel":
        return text

    return value


if "bayes_cv_svr" in locals() and hasattr(bayes_cv_svr, "best_params_"):
    best_svr_params = bayes_cv_svr.best_params_
else:
    best_svr_params = {}
    for col in [c for c in bayes_results_svr.columns if c.startswith("param_")]:
        name = col.replace("param_", "")
        parsed_value = _parse_svr_param(name, best_row[col])
        if parsed_value is not None:
            best_svr_params[name] = parsed_value

best_cv_rmse = -float(best_row["mean_test_neg_root_mean_squared_error"])
best_cv_r2 = float(best_row["mean_test_r2"])

print("Best SVR Parameters Found:", best_svr_params)
print(f"Best CV RMSE: {best_cv_rmse:.4f}")
print(f"Best CV R2:   {best_cv_r2:.4f}")

# Best-per-kernel summary
cv_summary_per_kernel = []
for kernel in sorted(bayes_results_svr["param_svr__kernel"].dropna().astype(str).unique()):
    df_kernel = bayes_results_svr[bayes_results_svr["param_svr__kernel"].astype(str) == kernel]
    idx = df_kernel["mean_test_neg_root_mean_squared_error"].idxmax()
    row = df_kernel.loc[idx]

    kernel_params = {
        c.replace("param_", ""): row[c]
        for c in df_kernel.columns
        if c.startswith("param_") and not pd.isna(row[c]) and row[c] != ""
    }

    summary = {
        "kernel": kernel,
        "best_cv_rmse": -row["mean_test_neg_root_mean_squared_error"],
        "std_cv_rmse": row["std_test_neg_root_mean_squared_error"],
        "best_cv_r2": row["mean_test_r2"],
        "std_cv_r2": row["std_test_r2"],
        "best_params": kernel_params,
    }
    cv_summary_per_kernel.append(summary)

    print(
        f"Kernel={kernel:>4s} | Best CV RMSE: {summary['best_cv_rmse']:.4f} "
        f"| R2: {summary['best_cv_r2']:.4f} | params: {kernel_params}"
    )

bayes_results_svr.head()


<p style="font-family: 'Courier New', Courier, monospace; font-size: 30px; font-weight: bold; color: blue;  text-align: left;">
Plot: best CV RMSE and its STD vs the Cost 
</p>

In [ ]:
# Load C-sweep CV results used for the SVR C-sensitivity figure
legacy_c_sweep_paths = [
    os.path.join("SVR", "Results", "svr_c_sweep_cv.csv"),
    os.path.join("Results", "svr_c_sweep_cv.csv"),
]

if os.path.exists(C_SWEEP_PATH):
    sweep_df = pd.read_csv(C_SWEEP_PATH)
else:
    sweep_df = None
    for legacy_path in legacy_c_sweep_paths:
        if os.path.exists(legacy_path):
            sweep_df = pd.read_csv(legacy_path)
            sweep_df.to_csv(C_SWEEP_PATH, index=False)
            print(f"Migrated legacy C-sweep results to: {C_SWEEP_PATH}")
            break

    if sweep_df is None:
        raise FileNotFoundError(
            f"No C-sweep results found. Provide {C_SWEEP_PATH} or rerun the C-sweep cell."
        )

sweep_df


In [ ]:
# Display SVR C-sweep RMSE and RMSE variability

if "log10_C" not in sweep_df.columns:
    sweep_df["log10_C"] = np.log10(sweep_df["C"].astype(float))

required_sweep_cols = ["log10_C", "rmse_mean", "rmse_std", "r2_mean", "r2_std"]
missing_sweep_cols = [c for c in required_sweep_cols if c not in sweep_df.columns]
if missing_sweep_cols:
    raise KeyError(f"Missing expected C-sweep columns: {missing_sweep_cols}")

tick_fontsize = 12
axis_labelsize = 15
legend_fontsize = 12
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "custom"
plt.rcParams["mathtext.rm"] = "Times New Roman"
sns.set_style("whitegrid")

sweep_df_all = sweep_df.copy().sort_values("log10_C").reset_index(drop=True)
x = np.arange(len(sweep_df_all))
bar_width = 0.35

fig, ax1 = plt.subplots(figsize=(7, 3.5))

bars1 = ax1.bar(
    x - bar_width / 2,
    sweep_df_all["rmse_mean"],
    bar_width,
    color="#1976d2",
    label="RMSE",
    edgecolor="black",
    linewidth=1,
    zorder=3,
)
ax1.set_xlabel(r"$\log_{10}(C)$", fontsize=axis_labelsize)
ax1.set_xticks(x)
ax1.set_xticklabels([f"{v:.0f}" for v in sweep_df_all["log10_C"]], fontsize=tick_fontsize)
ax1.tick_params(axis="y", labelcolor="#1976d2", labelsize=tick_fontsize)
ax1.grid(True, axis="y")

ax2 = ax1.twinx()
bars2 = ax2.bar(
    x + bar_width / 2,
    sweep_df_all["rmse_std"],
    bar_width,
    color="#d81b60",
    label="Std. of RMSE",
    edgecolor="black",
    linewidth=1,
    zorder=3,
)
ax2.tick_params(axis="y", labelcolor="#d81b60", labelsize=tick_fontsize)
ax2.grid(False)

handles = [
    plt.Rectangle((0, 0), 1, 1, facecolor="#1976d2", edgecolor="black", label="CV RMSE (dB)"),
    plt.Rectangle((0, 0), 1, 1, facecolor="#d81b60", edgecolor="black", label="Std. of RMSE (dB)"),
]
ax1.legend(handles=handles, loc="upper right", fontsize=legend_fontsize)

fig.tight_layout()
plt.show()


In [ ]:
# Plot and save the focused SVR C-sweep RMSE figure for log10(C) >= -1

if "log10_C" not in sweep_df.columns:
    sweep_df["log10_C"] = np.log10(sweep_df["C"].astype(float))

tick_fontsize   = 12
axis_labelsize  = 15
legend_fontsize = 12

plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "custom"
plt.rcParams["mathtext.rm"] = "Times New Roman"
sns.set_style("whitegrid")

sweep_df_plot = sweep_df.copy()
sweep_df_plot["log10_C"] = pd.to_numeric(sweep_df_plot["log10_C"], errors="coerce")
sweep_df_plot = sweep_df_plot[sweep_df_plot["log10_C"] >= -1].copy()
sweep_df_plot = sweep_df_plot.sort_values("log10_C").reset_index(drop=True)

if sweep_df_plot.empty:
    raise ValueError("No SVR C-sweep rows found for log10(C) >= -1.")

x = np.arange(len(sweep_df_plot))
bar_width = 0.40

fig, ax1 = plt.subplots(figsize=(7, 3.5))

bars1 = ax1.bar(
    x - bar_width / 2,
    sweep_df_plot["rmse_mean"],
    bar_width,
    color="#1976d2",
    edgecolor="black",
    linewidth=1,
    zorder=3,
)
for bar in bars1:
    bar.set_hatch("///")

ax1.set_xlabel(r"$\log_{10}(C)$", fontsize=axis_labelsize)
ax1.set_xticks(x)
ax1.set_xticklabels([f"{v:.0f}" for v in sweep_df_plot["log10_C"]], fontsize=tick_fontsize)
ax1.tick_params(axis="y", labelcolor="#1976d2", labelsize=tick_fontsize)
ax1.grid(True, axis="y")

ax2 = ax1.twinx()
bars2 = ax2.bar(
    x + bar_width / 2,
    sweep_df_plot["rmse_std"],
    bar_width,
    color="#d81b60",
    edgecolor="black",
    linewidth=1,
    zorder=3,
)
ax2.tick_params(axis="y", labelcolor="#d81b60", labelsize=tick_fontsize)
ax2.grid(False)

for ax in (ax1, ax2):
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor("black")
        spine.set_linewidth(1.2)

ax1.bar_label(bars1, fmt="%.2f", padding=3, fontsize=tick_fontsize)
ax2.bar_label(bars2, fmt="%.2f", padding=3, fontsize=tick_fontsize)

ax1.set_ylim(0, float(sweep_df_plot["rmse_mean"].max()) * 1.28)
ax2.set_ylim(0, float(sweep_df_plot["rmse_std"].max()) * 1.35)

handles = [
    plt.Rectangle((0, 0), 1, 1, facecolor="#1976d2", edgecolor="black", hatch="///", label="CV RMSE (dB)"),
    plt.Rectangle((0, 0), 1, 1, facecolor="#d81b60", edgecolor="black", label="Std. of RMSE (dB)"),
]
leg = ax1.legend(
    handles=handles,
    loc="upper right",
    bbox_to_anchor=(0.98, 0.98),
    borderaxespad=0.0,
    fontsize=legend_fontsize,
    frameon=True,
    framealpha=1.0,
)
leg.get_frame().set_edgecolor("black")
leg.get_frame().set_linewidth(1.2)
leg.get_frame().set_facecolor("white")

fig.tight_layout()
fig_path = os.path.join(FIGURES_DIR, "SVR_RMSE_STD_vs_log10C.png")
plt.savefig(fig_path, dpi=2000, bbox_inches="tight", pad_inches=0.03)
plt.show()
print(f"Saved figure: {fig_path}")


<p style="font-family: 'Courier New', Courier, monospace; font-size: 30px; font-weight: bold; color: blue;  text-align: left;">
Plot: best CV R² and its STD per cpst C
</p>

In [ ]:
# Display SVR C-sweep R2 and R2 variability

if "log10_C" not in sweep_df.columns:
    sweep_df["log10_C"] = np.log10(sweep_df["C"].astype(float))

tick_fontsize = 12
axis_labelsize = 15
legend_fontsize = 12
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "custom"
plt.rcParams["mathtext.rm"] = "Times New Roman"
sns.set_style("whitegrid")

sweep_df_all = sweep_df.copy().sort_values("log10_C").reset_index(drop=True)
x = np.arange(len(sweep_df_all))
bar_width = 0.35

fig, ax1 = plt.subplots(figsize=(7, 3.5))

bars1 = ax1.bar(
    x - bar_width / 2,
    sweep_df_all["r2_mean"],
    bar_width,
    color="#00bcd4",
    label="R2",
    edgecolor="black",
    linewidth=1,
    alpha=0.8,
)
ax1.set_xlabel(r"$\log_{10}(C)$", fontsize=axis_labelsize)
ax1.set_xticks(x)
ax1.set_xticklabels([f"{v:.0f}" for v in sweep_df_all["log10_C"]], fontsize=tick_fontsize)
ax1.tick_params(axis="y", labelcolor="#00bcd4", labelsize=tick_fontsize)
ax1.set_ylim(0, 1.01)
ax1.grid(True, axis="y")

ax2 = ax1.twinx()
bars2 = ax2.bar(
    x + bar_width / 2,
    sweep_df_all["r2_std"],
    bar_width,
    color="#388e3c",
    label="Std. of R2",
    edgecolor="black",
    linewidth=1,
    alpha=0.8,
)
ax2.tick_params(axis="y", labelcolor="#388e3c", labelsize=tick_fontsize)
ax2.grid(False)

handles = [
    plt.Rectangle((0, 0), 1, 1, facecolor="#00bcd4", edgecolor="black", label="CV R2"),
    plt.Rectangle((0, 0), 1, 1, facecolor="#388e3c", edgecolor="black", label="Std. of R2"),
]
ax1.legend(handles=handles, loc="upper right", fontsize=legend_fontsize)

fig.tight_layout()
plt.show()


<p style="font-family: 'Courier New', Courier, monospace; font-size: 30px; font-weight: bold; color: blue;  text-align: left;">
Evaluate best SVR on train/test; save model 
</p>

In [ ]:
# Train final model, evaluate on held-out test, and save model/residual artifacts

best_svr_model = create_svr_pipeline()
best_svr_model.set_params(**best_svr_params)
best_svr_model.fit(X_train_df, y_train)

print("Best SVR Parameters Used:", best_svr_params)
print("\nFinal SVR model trained on the training split currently loaded in this notebook.")

# Train/test predictions
y_train_pred = best_svr_model.predict(X_train_df)
y_test_pred = best_svr_model.predict(X_test_df)

# OOF predictions using the selected best parameters
y_pred_oof = np.full(len(y_train), np.nan, dtype=float)
cv_fold_ids = sorted(np.unique(fold_assignments[fold_assignments != -1]))

if not cv_fold_ids:
    raise ValueError("No validation fold labels found for OOF residual generation.")

for fold_num in cv_fold_ids:
    tr_idx = np.where(fold_assignments != fold_num)[0]
    val_idx = np.where(fold_assignments == fold_num)[0]

    fold_model = create_svr_pipeline()
    fold_model.set_params(**best_svr_params)
    fold_model.fit(X_train_df.iloc[tr_idx], y_train[tr_idx])
    y_pred_oof[val_idx] = fold_model.predict(X_train_df.iloc[val_idx])

mask = ~np.isnan(y_pred_oof)
PL_pred_oof = y_pred_oof[mask]
resid_oof = y_train[mask] - PL_pred_oof

if mask.sum() != len(y_train):
    print(f"OOF residuals generated for {mask.sum()} of {len(y_train)} training rows.")

# Metrics
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
oof_mse = mean_squared_error(y_train[mask], PL_pred_oof)

train_rmse = np.sqrt(train_mse)
test_rmse = np.sqrt(test_mse)
oof_rmse = np.sqrt(oof_mse)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
oof_r2 = r2_score(y_train[mask], PL_pred_oof)

test_mape = mean_absolute_percentage_error(y_test, y_test_pred)
test_median_ae = median_absolute_error(y_test, y_test_pred)

metrics_df = pd.DataFrame({
    "Metric": [
        "Train MSE", "Train RMSE", "Train R2",
        "OOF MSE", "OOF RMSE", "OOF R2",
        "Test MSE", "Test RMSE", "Test R2",
        "Test MAPE (%)", "Test Median AE",
    ],
    "Value": [
        train_mse, train_rmse, train_r2,
        oof_mse, oof_rmse, oof_r2,
        test_mse, test_rmse, test_r2,
        test_mape * 100, test_median_ae,
    ],
})

print("\nModel Evaluation Metrics:")
display(metrics_df)

# Save final model
with open(MODEL_PATH, "wb") as f:
    pickle.dump(best_svr_model, f)
print(f"\nTrained SVR model saved to: {MODEL_PATH}")

# TEST residuals
PL_pred_test = y_test_pred
resid_test = y_test - PL_pred_test

svr_test_df = pd.DataFrame({
    "model":       "SVR",
    "split":       "test",
    "row_id":      np.arange(len(df_test), dtype=int),
    "time":        df_test.get("time", pd.Series(index=df_test.index, dtype=float)).values,
    "device_id":   df_test["device_id"].values,
    "distance":    df_test["distance"].values,
    "frequency":   df_test["frequency"].values,
    "c_walls":     df_test["c_walls"].values,
    "w_walls":     df_test["w_walls"].values,
    "co2":         df_test["co2"].values,
    "humidity":    df_test["humidity"].values,
    "pm25":        df_test["pm25"].values,
    "pressure":    df_test["pressure"].values,
    "temperature": df_test["temperature"].values,
    "snr":         df_test["snr"].values,
    "PL_true":     y_test,
    "PL_pred":     PL_pred_test,
    "resid_db":    resid_test,
})

test_residual_path = os.path.join(RESIDUALS_DIR, "residuals_SVR_test.csv")
svr_test_df.to_csv(test_residual_path, index=False)
print(f"[TEST] Saved SVR test residuals: {test_residual_path}")

# OOF residuals
svr_oof_df = pd.DataFrame({
    "model":       "SVR",
    "split":       "oof",
    "row_id":      np.arange(len(df_train), dtype=int)[mask],
    "fold":        fold_assignments.astype(int)[mask],
    "time":        df_train.get("time", pd.Series(index=df_train.index, dtype=float)).values[mask],
    "device_id":   df_train["device_id"].values[mask],
    "distance":    df_train["distance"].values[mask],
    "frequency":   df_train["frequency"].values[mask],
    "c_walls":     df_train["c_walls"].values[mask],
    "w_walls":     df_train["w_walls"].values[mask],
    "co2":         df_train["co2"].values[mask],
    "humidity":    df_train["humidity"].values[mask],
    "pm25":        df_train["pm25"].values[mask],
    "pressure":    df_train["pressure"].values[mask],
    "temperature": df_train["temperature"].values[mask],
    "snr":         df_train["snr"].values[mask],
    "PL_true":     y_train[mask],
    "PL_pred":     PL_pred_oof,
    "resid_db":    resid_oof,
})

oof_residual_path = os.path.join(RESIDUALS_DIR, "residuals_SVR_oof.csv")
svr_oof_df.to_csv(oof_residual_path, index=False)
print(f"[OOF] Saved SVR OOF residuals: {oof_residual_path}")


<p style="font-family: 'Courier New', Courier, monospace; font-size: 30px; font-weight: bold; color: blue;  text-align: left;">
Diagnostics: learning curve, residuals, physics, predicted vs. real
</p>


In [ ]:
# Final SVR diagnostic plots

svr_model = best_svr_model
figsize = (7, 3)

# 1. Learning Curve
train_sizes, train_scores, test_scores = learning_curve(
    svr_model,
    X_train_df,
    y_train,
    cv=PredefinedSplit(fold_assignments),
    n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 5),
    scoring="neg_root_mean_squared_error",
)
train_rmse_curve = -train_scores.mean(axis=1)
val_rmse_curve = -test_scores.mean(axis=1)

plt.figure(figsize=figsize)
plt.plot(train_sizes, train_rmse_curve, "o-", label="Train RMSE")
plt.plot(train_sizes, val_rmse_curve, "o-", label="CV RMSE")
plt.xlabel("Training Size")
plt.ylabel("RMSE (dB)")
plt.title("SVR Learning Curve")
plt.legend()
plt.show()

# 2. Held-out residuals
plt.figure(figsize=figsize)
plt.scatter(y_test_pred, resid_test, alpha=0.5)
plt.axhline(0, color="r", linestyle="--")
plt.xlabel("Predicted PL (dB)")
plt.ylabel("Residuals (dB)")
plt.title("SVR Held-out Residuals")
plt.show()

# 3. Physics consistency: PL vs. log(distance)
dist = df_test["distance"].to_numpy()
log_dist = np.log10(dist + 1e-6)
slope, intercept, r_value, _, _ = linregress(log_dist, y_test_pred)

plt.figure(figsize=figsize)
plt.scatter(log_dist, y_test_pred, alpha=0.5)
plt.plot(log_dist, intercept + slope * log_dist, "r--", label=f"n ~ {slope:.2f}, R2={r_value**2:.2f}")
plt.xlabel("log10(Distance)")
plt.ylabel("Predicted PL (dB)")
plt.title("SVR PL vs. log(Distance)")
plt.legend()
plt.show()

# 4. Predicted vs. real PL
plt.figure(figsize=figsize)
lims = [min(y_test.min(), y_test_pred.min()), max(y_test.max(), y_test_pred.max())]
plt.scatter(y_test, y_test_pred, alpha=0.5)
plt.plot(lims, lims, "r--")
plt.xlabel("Real PL (dB)")
plt.ylabel("Predicted PL (dB)")
plt.title("SVR Predicted vs. Real")
plt.show()


<p style="font-family: 'Courier New', Courier, monospace; font-size: 30px; font-weight: bold; color: blue;  text-align: left;">
SVR model complexity diagnostic
</p>

In [ ]:
# Support-vector count as an SVR model-complexity diagnostic
try:
    n_sv = best_svr_model.named_steps["svr"].support_.shape[0]
    frac_sv = n_sv / X_train_df.shape[0]
    print(f"Support vectors: {n_sv} ({frac_sv:.2%} of training samples)")
except Exception as exc:
    print("Could not compute support vector count:", exc)
